In [ ]:
# Cell 1 — File listing
import numpy as np
import pandas as pd
import os
from pathlib import Path

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

print('\n✓ Cell 1 complete — file listing done')

In [ ]:
# Cell 2 — Install dependencies
import subprocess
result = subprocess.run(
    ['pip', 'install', 'timm', 'librosa', '--quiet'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '')
print(result.stderr[-200:] if result.stderr else '')
print('✓ Cell 2 complete — packages installed')

In [ ]:
# Cell 3 — All imports
import os, gc, time, warnings, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
from collections import Counter

warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'  numpy   : {np.__version__}')
print(f'  pandas  : {pd.__version__}')
print(f'  torch   : {torch.__version__}')
print(f'  librosa : {librosa.__version__}')
print(f'  timm    : {timm.__version__}')
print(f'  device  : {DEVICE}')
print('✓ Cell 3 complete — all imports successful')

In [ ]:
# Cell 4 — Configuration
class CFG:
    # Paths
    base_dir        = Path('/kaggle/input/competitions/birdclef-2026')
    output_dir      = Path('/kaggle/working')

    # Audio
    sr              = 32_000
    duration        = 5
    n_samples       = 32_000 * 5      # 160_000
    n_mels          = 128
    hop_length      = 320
    fmin            = 50
    fmax            = 16_000
    n_fft           = 1024

    # Training
    epochs          = 15
    batch_size      = 32
    lr              = 1e-3
    weight_decay    = 1e-4
    label_smoothing = 0.05
    mixup_alpha     = 0.3
    secondary_weight= 0.5
    min_rating      = 3.0

    # Augmentation
    time_mask_param = 40
    freq_mask_param = 20

    # Model
    model_name      = 'efficientnet_b0'
    pretrained      = True
    num_workers     = 2

    # Inference
    prob_clip_low   = 0.01
    prob_clip_high  = 0.99
    tta             = True

# Audio directory paths (plain variables, not inside class)
train_audio_dir = CFG.base_dir / 'train_audio'
train_sc_dir    = CFG.base_dir / 'train_soundscapes'
test_sc_dir     = CFG.base_dir / 'test_soundscapes'

print(f'  base_dir       : {CFG.base_dir}')
print(f'  output_dir     : {CFG.output_dir}')
print(f'  train_audio_dir: {train_audio_dir}')
print(f'  train_sc_dir   : {train_sc_dir}')
print(f'  test_sc_dir    : {test_sc_dir}')
print(f'  n_samples      : {CFG.n_samples}')
print(f'  device         : {DEVICE}')
print('✓ Cell 4 complete — config ready')

In [ ]:
# Cell 5 — Load CSVs & detect audio
train_df   = pd.read_csv(CFG.base_dir / 'train.csv')
taxonomy   = pd.read_csv(CFG.base_dir / 'taxonomy.csv')
sc_labels  = pd.read_csv(CFG.base_dir / 'train_soundscapes_labels.csv')
sample_sub = pd.read_csv(CFG.base_dir / 'sample_submission.csv')

# Species list from submission
SPECIES        = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES    = len(SPECIES)
SPECIES_TO_IDX = {s: i for i, s in enumerate(SPECIES)}

print(f'  train_df rows       : {len(train_df):,}')
print(f'  taxonomy rows       : {len(taxonomy):,}')
print(f'  sc_labels rows      : {len(sc_labels):,}')
print(f'  sample_sub shape    : {sample_sub.shape}')
print(f'  NUM_CLASSES         : {NUM_CLASSES}')
print()
print(f'  sc_labels columns   : {list(sc_labels.columns)}')
print(f'  sc_labels sample    :\n{sc_labels.head(3).to_string()}')
print()

# Detect audio directories
has_train_audio = train_audio_dir.exists() and any(train_audio_dir.rglob('*.ogg'))
has_train_sc    = train_sc_dir.exists()    and any(train_sc_dir.glob('*.ogg'))
has_test_sc     = test_sc_dir.exists()     and any(test_sc_dir.glob('*.ogg'))

train_sc_files    = sorted(train_sc_dir.glob('*.ogg'))    if has_train_sc    else []
test_sc_files     = sorted(test_sc_dir.glob('*.ogg'))     if has_test_sc     else []
train_audio_files = list(train_audio_dir.rglob('*.ogg'))  if has_train_audio else []

print(f'  train_audio/       found : {has_train_audio}  ({len(train_audio_files)} files)')
print(f'  train_soundscapes/ found : {has_train_sc}  ({len(train_sc_files)} files)')
print(f'  test_soundscapes/  found : {has_test_sc}  ({len(test_sc_files)} files)')

if not has_train_audio and not has_train_sc and not has_test_sc:
    print()
    print('  ⚠️  No audio found. Go to:')
    print('  Notebook → Add Data → Competitions → birdclef-2026')

print('✓ Cell 5 complete — CSVs loaded, audio presence detected')

In [ ]:
# Cell 6 — Audio utilities
def parse_start_seconds(val):
    """
    Convert any time format to float seconds.
    Handles: '00:00:05', '0:05', '5', 5, 5.0, '00:00:05.500'
    """
    if isinstance(val, (int, float)):
        return float(val)
    val = str(val).strip()
    parts = val.split(':')
    if len(parts) == 3:
        h, m, s = parts
        return int(h) * 3600 + int(m) * 60 + float(s)
    elif len(parts) == 2:
        m, s = parts
        return int(m) * 60 + float(s)
    else:
        return float(val)


def load_clip(path, start_sec, duration_sec=CFG.duration, sr=CFG.sr):
    """Load exactly duration_sec seconds starting at start_sec."""
    y, _ = librosa.load(
        str(path), sr=sr, mono=True,
        offset=float(start_sec),
        duration=float(duration_sec)
    )
    needed = int(duration_sec * sr)
    if len(y) < needed:
        y = np.pad(y, (0, needed - len(y)))
    return y[:needed]


def load_full(path, sr=CFG.sr):
    """Load entire audio file."""
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    return y


def to_melspec(y):
    """1-D audio array → normalised log-mel spectrogram float32 (H, W)."""
    needed = CFG.n_samples
    if len(y) < needed:
        y = np.pad(y, (0, needed - len(y)))
    else:
        y = y[:needed]
    mel = librosa.feature.melspectrogram(
        y=y, sr=CFG.sr,
        n_mels=CFG.n_mels,
        hop_length=CFG.hop_length,
        n_fft=CFG.n_fft,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    lo, hi = mel_db.min(), mel_db.max()
    mel_db = (mel_db - lo) / (hi - lo + 1e-6)
    return mel_db.astype(np.float32)   # (128, 500)


def spec_augment(spec):
    """SpecAugment: random time + frequency masking."""
    spec = spec.copy()
    _, T = spec.shape
    f0 = random.randint(0, CFG.n_mels - CFG.freq_mask_param)
    spec[f0:f0 + CFG.freq_mask_param, :] = 0.0
    t0 = random.randint(0, T - CFG.time_mask_param)
    spec[:, t0:t0 + CFG.time_mask_param] = 0.0
    return spec


# Smoke test
_dummy = np.zeros(CFG.n_samples, dtype=np.float32)
_spec  = to_melspec(_dummy)
print(f'  to_melspec output shape : {_spec.shape}  (expected (128, 500))')
print(f'  parse_start_seconds tests:')
for v in ['00:00:05', '0:05', '5', 5, '00:01:30', '00:00:05.500']:
    print(f"    '{v}' → {parse_start_seconds(v):.2f}s")
print('✓ Cell 6 complete — audio utilities ready')

In [ ]:
# Cell 7 — Label builders
def build_sc_label_vector(species_str):
    """Semicolon-separated species string → float32 label vector."""
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if not isinstance(species_str, str):
        return vec
    for sp in species_str.split(';'):
        sp = sp.strip()
        if sp in SPECIES_TO_IDX:
            vec[SPECIES_TO_IDX[sp]] = 1.0
    return vec


def build_train_label_vector(primary, secondary_str, w=CFG.secondary_weight):
    """Primary + secondary labels → float32 label vector."""
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if primary in SPECIES_TO_IDX:
        vec[SPECIES_TO_IDX[primary]] = 1.0
    if isinstance(secondary_str, str):
        cleaned = (
            secondary_str
            .replace('[', '').replace(']', '')
            .replace("'", '').replace('"', '')
        )
        for sp in cleaned.split(','):
            sp = sp.strip()
            if sp in SPECIES_TO_IDX:
                vec[SPECIES_TO_IDX[sp]] = max(vec[SPECIES_TO_IDX[sp]], w)
    return vec


# Smoke tests
_v1 = build_sc_label_vector(sc_labels['primary_label'].iloc[0])
_v2 = build_train_label_vector(
    train_df['primary_label'].iloc[0],
    train_df['secondary_labels'].iloc[0]
)
print(f'  sc label vector    — sum={_v1.sum():.0f}  nonzero={np.nonzero(_v1)[0].tolist()}')
print(f'  train label vector — sum={_v2.sum():.1f}  nonzero={np.nonzero(_v2)[0].tolist()}')
print('✓ Cell 7 complete — label builders ready')

In [ ]:
# Cell 8 — Build records
def build_records():
    records = []

    # ── Labeled soundscape segments ───────────────────────────────────────
    if has_train_sc and len(sc_labels) > 0:
        missing_sc, added_sc, bad_start = 0, 0, 0
        for _, row in sc_labels.iterrows():
            path = train_sc_dir / row['filename']
            if not path.exists():
                missing_sc += 1
                continue
            label = build_sc_label_vector(row['primary_label'])
            if label.sum() == 0:
                continue
            try:
                start = parse_start_seconds(row['start'])
            except Exception:
                bad_start += 1
                continue
            records.append({
                'path'  : str(path),
                'label' : label,
                'start' : start,
                'source': 'soundscape_labeled',
            })
            added_sc += 1
        print(f'  Soundscape labeled  added    : {added_sc:,}')
        print(f'  Soundscape labeled  missing  : {missing_sc:,}')
        print(f'  Soundscape labeled  bad_start: {bad_start:,}')
    else:
        print('  No labeled soundscape segments available (files missing).')

    # ── Individual train_audio recordings ─────────────────────────────────
    sc_count = len(records)
    if has_train_audio:
        skipped, added_audio = 0, 0
        for _, row in train_df.iterrows():
            primary = row['primary_label']
            if primary not in SPECIES_TO_IDX:
                skipped += 1
                continue
            try:
                rating = float(row.get('rating', 0))
            except (ValueError, TypeError):
                rating = 0.0
            if row.get('collection') == 'XC' and 0 < rating < CFG.min_rating:
                skipped += 1
                continue
            path = train_audio_dir / row['filename']
            if not path.exists():
                alt = train_audio_dir / Path(row['filename']).name
                if not alt.exists():
                    skipped += 1
                    continue
                path = alt
            label = build_train_label_vector(
                primary, row.get('secondary_labels', ''))
            records.append({
                'path'  : str(path),
                'label' : label,
                'start' : None,
                'source': 'train_audio',
            })
            added_audio += 1
        print(f'  train_audio added   : {added_audio:,}')
        print(f'  train_audio skipped : {skipped:,}')
    else:
        print('  train_audio/ not available — skipped.')

    # ── Summary ───────────────────────────────────────────────────────────
    print()
    print(f'  Total records : {len(records):,}')

    if len(records) == 0:
        print()
        print('  ⚠️  ZERO RECORDS — no audio found.')
        print('  Go to Notebook → Add Data → Competitions → birdclef-2026')
        print('  Minimum needed: train_soundscapes/ folder (~2 GB)')
    else:
        for src, cnt in Counter(r['source'] for r in records).items():
            print(f"  source '{src}': {cnt:,} records")
        all_labels = np.stack([r['label'] for r in records])
        covered    = int((all_labels.sum(axis=0) > 0).sum())
        print(f'  Species covered : {covered} / {NUM_CLASSES}')

    return records


records = build_records()
print('✓ Cell 8 complete — records built')

In [ ]:
# Cell 9 — Dataset class
class BirdDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def _get_audio(self, rec):
        if rec['start'] is not None:
            return load_clip(rec['path'], rec['start'])
        else:
            y = load_full(rec['path'])
            if len(y) > CFG.n_samples:
                s = random.randint(0, len(y) - CFG.n_samples)
                y = y[s:s + CFG.n_samples]
            return y

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            y    = self._get_audio(rec)
            spec = to_melspec(y)
        except Exception:
            spec = np.zeros(
                (CFG.n_mels, CFG.n_samples // CFG.hop_length),
                dtype=np.float32)
        if self.augment:
            spec = spec_augment(spec)
        x = torch.tensor(spec).unsqueeze(0)           # (1, 128, 500)
        y = torch.tensor(rec['label'], dtype=torch.float32)
        return x, y


# Smoke test — only if records exist
if len(records) > 0:
    _ds      = BirdDataset(records[:4], augment=False)
    _x, _y  = _ds[0]
    print(f'  Dataset length   : {len(_ds)}')
    print(f'  Sample x shape   : {_x.shape}   (expected torch.Size([1, 128, 500]))')
    print(f'  Sample y shape   : {_y.shape}   (expected torch.Size([{NUM_CLASSES}]))')
    print(f'  Sample y sum     : {_y.sum().item():.1f}')
else:
    print('  ⚠ No records — dataset smoke test skipped.')
print('✓ Cell 9 complete — BirdDataset ready')

In [ ]:
# Cell 10 — Model
class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            CFG.model_name,
            pretrained=CFG.pretrained,
            in_chans=1,
            num_classes=0,
            global_pool='avg',
        )
        feat_dim = self.encoder.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, NUM_CLASSES),
        )

    def forward(self, x):
        return self.head(self.encoder(x))


class SmoothBCE(nn.Module):
    def __init__(self, eps=CFG.label_smoothing):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        targets = targets * (1 - self.eps) + 0.5 * self.eps
        return F.binary_cross_entropy_with_logits(logits, targets)


model    = BirdModel().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())

# Forward pass smoke test
_dummy_x   = torch.zeros(2, 1, CFG.n_mels,
                          CFG.n_samples // CFG.hop_length).to(DEVICE)
_dummy_out = model(_dummy_x)
print(f'  Model         : {CFG.model_name}')
print(f'  Parameters    : {n_params:,}')
print(f'  Input shape   : {tuple(_dummy_x.shape)}')
print(f'  Output shape  : {tuple(_dummy_out.shape)}  (expected (2, {NUM_CLASSES}))')
del _dummy_x, _dummy_out
print('✓ Cell 10 complete — model ready')

In [ ]:
# Cell 11 — Training loop
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_n = 0.0, 0
    for x, y in tqdm(loader, leave=False, desc='  train'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        # Mixup (50% of batches)
        if CFG.mixup_alpha > 0 and random.random() < 0.5:
            idx = torch.randperm(x.size(0))
            lam = float(np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha))
            x   = lam * x + (1 - lam) * x[idx]
            y   = lam * y + (1 - lam) * y[idx]
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_n    += x.size(0)
    return total_loss / total_n


@torch.no_grad()
def validate(model, loader):
    model.eval()
    probs_list, tgts_list = [], []
    for x, y in tqdm(loader, leave=False, desc='  valid'):
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
        probs_list.append(p)
        tgts_list.append(y.numpy())
    probs   = np.concatenate(probs_list)
    targets = np.concatenate(tgts_list)
    aucs = []
    for i in range(NUM_CLASSES):
        if targets[:, i].sum() > 0:
            try:
                aucs.append(roc_auc_score(targets[:, i], probs[:, i]))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else 0.0


def run_training(model, records):
    random.shuffle(records)
    n_val    = max(1, int(len(records) * 0.1))
    val_r    = records[:n_val]
    train_r  = records[n_val:]

    train_ds = BirdDataset(train_r, augment=True)
    val_ds   = BirdDataset(val_r,   augment=False)
    train_ld = DataLoader(train_ds, batch_size=CFG.batch_size,
                          shuffle=True,  num_workers=CFG.num_workers,
                          pin_memory=False)
    val_ld   = DataLoader(val_ds,   batch_size=CFG.batch_size,
                          shuffle=False, num_workers=CFG.num_workers,
                          pin_memory=False)

    criterion = SmoothBCE()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.epochs, eta_min=1e-6)

    best_auc  = 0.0
    best_path = CFG.output_dir / 'best_model.pt'
    history   = []

    print(f'  train samples : {len(train_r):,}')
    print(f'  val   samples : {len(val_r):,}')
    print(f'  epochs        : {CFG.epochs}')
    print()

    for epoch in range(1, CFG.epochs + 1):
        t0      = time.time()
        loss    = train_one_epoch(model, train_ld, optimizer, criterion)
        auc     = validate(model, val_ld)
        scheduler.step()
        elapsed = time.time() - t0
        marker  = ' ← best' if auc > best_auc else ''
        print(f'  Epoch {epoch:02d}/{CFG.epochs} | '
              f'loss={loss:.4f} | val_auc={auc:.4f} | {elapsed:.0f}s{marker}')
        history.append({'epoch': epoch, 'loss': loss, 'val_auc': auc})
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f'\n  Best val ROC-AUC : {best_auc:.4f}')
    print(f'  Best model saved : {best_path}')
    return model, history


print('✓ Cell 11 complete — training functions defined')

In [ ]:
# Cell 12 — Run training or skip
history = []

if len(records) == 0:
    print('⚠️  No records — skipping training.')
    print('   Model stays at random initialisation.')
    print('   Submission will use species-frequency priors.')
else:
    print(f'Starting training on {DEVICE} ...')
    model, history = run_training(model, records)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print()
    print(f'Training history ({len(history)} epochs):')
    for h in history:
        print(f"  epoch={h['epoch']:02d}  loss={h['loss']:.4f}  val_auc={h['val_auc']:.4f}")

print('✓ Cell 12 complete — training done')

In [ ]:
# Cell 13 — Inference utilities
class TestDataset(Dataset):
    def __init__(self, segments, shift=0):
        self.segments = segments
        self.shift    = shift       # sample shift for TTA

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        seg   = self.segments[idx]
        start = max(0.0, float(seg['start']) + self.shift / CFG.sr)
        try:
            y    = load_clip(seg['path'], start)
            spec = to_melspec(y)
        except Exception:
            spec = np.zeros(
                (CFG.n_mels, CFG.n_samples // CFG.hop_length),
                dtype=np.float32)
        return torch.tensor(spec).unsqueeze(0), seg['row_id']


def build_test_segments():
    """One entry per 5-second segment of every test soundscape."""
    segs = []
    src  = test_sc_dir if has_test_sc else None
    if src is None:
        print('  ⚠ test_soundscapes/ not found — using sample_submission row_ids.')
        return segs
    for fpath in sorted(src.glob('*.ogg')):
        stem = fpath.stem
        for end in range(5, 65, 5):
            segs.append({
                'path'  : str(fpath),
                'start' : float(end - 5),
                'row_id': f'{stem}_{end}',
            })
    return segs


@torch.no_grad()
def infer(model, segments, shift=0):
    model.eval()
    ds  = TestDataset(segments, shift=shift)
    ld  = DataLoader(ds, batch_size=64, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=False)
    all_p, all_ids = [], []
    for x, ids in tqdm(ld, desc=f'  infer shift={shift}', leave=False):
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
        all_p.append(p)
        all_ids.extend(ids)
    return np.concatenate(all_p), all_ids


test_segments = build_test_segments()
print(f'  Test soundscape files   : {len(test_sc_files)}')
print(f'  Test segments (5s each) : {len(test_segments):,}')
print(f'  Expected rows           : {len(test_sc_files) * 12}')
print('✓ Cell 13 complete — inference utilities ready')

In [ ]:
# Cell 14 — Run inference
def make_prior_submission(row_ids):
    """Species-frequency prior when no model is trained."""
    freq = np.full(NUM_CLASSES, CFG.prob_clip_low, dtype=np.float32)
    if len(sc_labels) > 0:
        for _, row in sc_labels.iterrows():
            freq += build_sc_label_vector(row['primary_label'])
        freq = freq / (len(sc_labels) + 1)
    freq = np.clip(freq, CFG.prob_clip_low, CFG.prob_clip_high)
    return np.tile(freq, (len(row_ids), 1))


if len(test_segments) == 0:
    print('No test segments — building prior submission from sample_submission row_ids.')
    row_ids     = list(sample_sub['row_id'])
    probs_final = make_prior_submission(row_ids)

elif len(records) == 0:
    print('No training records — building prior submission.')
    row_ids     = [s['row_id'] for s in test_segments]
    probs_final = make_prior_submission(row_ids)

else:
    print('Running model inference ...')

    # Base pass
    probs_base, row_ids = infer(model, test_segments, shift=0)
    print(f'  Base inference done  — shape {probs_base.shape}')

    if CFG.tta:
        half      = CFG.sr // 2          # 0.5 s shift
        p_fwd, _ = infer(model, test_segments, shift= half)
        p_bwd, _ = infer(model, test_segments, shift=-half)
        probs_final = (probs_base + p_fwd + p_bwd) / 3.0
        print(f'  TTA (3x) done        — shape {probs_final.shape}')
    else:
        probs_final = probs_base

probs_final = np.clip(probs_final, CFG.prob_clip_low, CFG.prob_clip_high)

print(f'  row_ids count  : {len(row_ids):,}')
print(f'  probs shape    : {probs_final.shape}')
print(f'  prob min/max   : {probs_final.min():.4f} / {probs_final.max():.4f}')
print('✓ Cell 14 complete — inference done')

In [ ]:
# Cell 15 — Build & save submission
sub = pd.DataFrame(probs_final, columns=SPECIES)
sub.insert(0, 'row_id', row_ids)

# Add any columns missing from submission (fill with prior)
for col in sample_sub.columns:
    if col not in sub.columns:
        sub[col] = CFG.prob_clip_low

# Align to exact sample_submission column order
sub = sub[sample_sub.columns]

# Save
out_path = CFG.output_dir / 'submission.csv'
sub.to_csv(out_path, index=False)

print(f'  Output path    : {out_path}')
print(f'  Shape          : {sub.shape}  (expected rows x {len(sample_sub.columns)} cols)')
print(f'  row_id[0]      : {sub["row_id"].iloc[0]}')
print(f'  row_id[-1]     : {sub["row_id"].iloc[-1]}')
print(f'  Prob stats:\n{sub[SPECIES].stack().describe().round(4).to_string()}')
print('✓ Cell 15 complete — submission.csv saved')

In [ ]:
# Cell 16 — Final validation checks
errors = []

# Shape
expected_cols = len(sample_sub.columns)
if sub.shape[1] != expected_cols:
    errors.append(f'Column count {sub.shape[1]} ≠ expected {expected_cols}')

# row_id column present
if 'row_id' not in sub.columns:
    errors.append("Missing 'row_id' column")

# No NaN
if sub[SPECIES].isnull().any().any():
    errors.append('NaN values found in predictions')

# No Inf
if np.isinf(sub[SPECIES].values).any():
    errors.append('Inf values found in predictions')

# Prob range
lo = sub[SPECIES].min().min()
hi = sub[SPECIES].max().max()
if lo < 0 or hi > 1:
    errors.append(f'Probabilities out of [0,1]: min={lo}, max={hi}')

# Column order matches sample_submission
if list(sub.columns) != list(sample_sub.columns):
    errors.append('Column order does not match sample_submission')

print('=' * 55)
if errors:
    for e in errors:
        print(f'  ✗ {e}')
    print('=' * 55)
    print('⚠️  Submission has issues — fix before submitting.')
else:
    print(f'  ✓ Shape correct      : {sub.shape}')
    print('  ✓ No NaN / Inf')
    print(f'  ✓ Probabilities in   : [{lo:.4f}, {hi:.4f}]')
    print('  ✓ Column order matches sample_submission')
    print('  ✓ submission.csv is valid and ready to submit')
print('=' * 55)
print('✓ Cell 16 complete — all checks passed')